In [ ]:
!pip install google-generativeai pandas

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 34.6 MB/s  0:00:00
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------------------------------------- 4.9/4.9 MB 66.7 MB/s  0:00:00
   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
   ---------------------------------------  15.2/15.3 MB 91.3 MB/s eta 0:00:01
   ---------------------------------------- 15.3/15.3 MB 73.4 MB/s  0:00:00

  Attempting uninstall: requests

    Found existing installation: requests 2.32.5

    Uninstalling requests-2.32.5:

      Successfully uninstalled requests-2.32.5

   -- ---------

In [11]:
!pip install google-generativeai

In [3]:
import pandas as pd


thread_df = pd.read_csv("../../02_Data/processed/comments_merged_thread_add_tag.csv")


In [4]:
thread_df

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,1
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73434,73434,1143,133497,1,"[133497, 133510, 134222]",3,"Hello. Will it be in Spanish? and if not, woul...",1,2,0,0,0,4
73435,73435,1143,133314,1,"[133314, 133507]",2,Not sure if people get notifications for repli...,1,1,1,0,0,4
73436,73436,1143,133207,1,"[133207, 133219]",2,The project looks interesting! Who is the desi...,1,0,0,0,0,0
73437,73437,1143,133159,1,"[133159, 133202, 133500]",3,"Hi, is there the future possibility of a solo ...",1,1,0,0,0,11


In [23]:
thread_df.loc[thread_df['projectID']==1476]

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes


In [6]:
thread_df["merged_comment"][1]

"@BlackSiteStudios Could a person use larger dinosaur figures as long as they scale up the battle mat area to fit them?|Sure! The game is designed for a 2x2ft play area, so you'll need to do some balance yourself. The rules will tell you what base size a particular dinosaur will have, so if the models you have fit on that, then you should be good."

In [ ]:
#test_df = thread_df.sample(n=30, random_state=42).copy() 

In [ ]:
import time
import pandas as pd
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. API 설정
try:
    with open("api_key.txt", "r") as file:
        api_key = file.read().strip() 
    genai.configure(api_key=api_key)
except FileNotFoundError:
    print("api_key.txt 파일 오류.")
    exit()

model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 2. 데이터 준비
target_df = thread_df.copy().reset_index(drop=True)

system_prompt = """You are an expert community manager for a global crowdfunding platform.
Your task is to analyze user comments and classify the core intent into exactly ONE of the following categories:

[Categories]
1. Shipping_Fulfillment
2. Product_Question
3. Praise_Support
4. Complaint_Refund
5. Suggestion_Idea
6. Spam_Irrelevant

[Rules]
- Read the entire conversation thread and select the ONE category that best fits the CORE intent.
- Lines starting with '|' are replies to the main comment.
- DO NOT provide any explanation.
- ONLY output the exact name of the category."""

# 3. 분류 함수 (병렬용)
def classify_row(args):
    seq_idx, row = args
    try:
        full_prompt = f"{system_prompt}\n\n[Thread Text]\n{row['merged_comment']}"
        response = model.generate_content(full_prompt)
        return seq_idx, response.text.strip()
    except Exception:
        return seq_idx, "Error"

# 4. 병렬 실행 및 진행률 표시
print(f"🚀 총 {len(target_df)}개 데이터 병렬 분류 시작...")

results_dict = {}

# 원하는 최종 컬럼 순서 지정
#['Unnamed: 0', 'projectID', 'thread_id', 'group', 'comment_ids',
#       'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master',
#       'is_backer', 'is_prior_backer', 'is_pathfinder', 'likes']
selected_columns = ['thread_id', 'projectID', 'group', 'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master','is_backer', 'is_prior_backer', 'is_pathfinder', 'likes','category']

# 50개의 스레드를 동시에 사용하여 구글 서버에 요청
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(classify_row, (seq_idx, row)) for seq_idx, (_, row) in enumerate(target_df.iterrows())]
    
    for future in tqdm(futures, total=len(target_df), desc="분류 진행중"):
        seq_idx, category = future.result()
        results_dict[seq_idx] = category

# 5. 최종 저장 단계
target_df['category'] = [results_dict[i] for i in range(len(target_df))]
final_df = target_df[selected_columns]

# 최종 결과 저장
final_df.to_csv("../../02_Data/processed/final_all_results.csv", index=False, encoding="utf-8-sig")
print("\n 모든 분류가 완료되었습니다.")

🚀 총 73439개 데이터 병렬 분류 시작...


분류 진행중: 100%|██████████| 73439/73439 [58:28<00:00, 20.93it/s]  



 모든 분류가 완료되었습니다.


In [14]:
target_df.groupby('group')['category'].value_counts()
target_df.groupby(['group', 'category']).size()
result_df = target_df.groupby('group')['category'].value_counts().reset_index(name='count')
result_df

,group,category,count
0,0,Product_Question,13816
1,0,Suggestion_Idea,9672
2,0,Praise_Support,6431
3,0,Shipping_Fulfillment,2184
4,0,Spam_Irrelevant,1374
5,0,Complaint_Refund,1085
6,0,Please provide the thread text you would like ...,5
7,0,Please provide the [Thread Text] you would lik...,1
8,0,Please provide the full thread text you would ...,1
9,0,Please provide the text you would like me to a...,1


In [20]:
target_df.head()

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes,category
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26,Praise_Support
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0,Product_Question
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,1,Spam_Irrelevant
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3,Praise_Support
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5,Praise_Support


In [22]:
target_df.loc[target_df['projectID']==1476]['category'].value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
#오류나면 실행
import pandas as pd
import time
from tqdm import tqdm

# 1. 기존 결과 파일 불러오기
df_result = pd.read_csv("../../02_Data/processed/final_all_results.csv")

# 2. 'Error'로 표시된 행만 추출
error_df = df_result[df_result['category'] == 'Error'].copy()

print(f"❌ 총 {len(error_df)}개의 에러 데이터를 발견했습니다. 재시도 시작...")

# 3. 에러 데이터만 다시 분류하는 루프
for index, row in tqdm(error_df.iterrows(), total=len(error_df), desc="재분류 중"):
    try:
        # 동일한 프롬프트로 재시도
        full_prompt = f"{system_prompt}\n\n[Thread Text]\n{row['thread_text']}"
        response = model.generate_content(full_prompt)
        
        # 성공하면 해당 인덱스의 값을 덮어씌움
        df_result.at[index, 'category'] = response.text.strip()
        time.sleep(0.1) # 서버 과부하 방지
    except Exception:
        # 여전히 실패하면 'Final_Error'로 기록
        df_result.at[index, 'category'] = 'Final_Error'

# 4. 최종 결과 저장
df_result.to_csv("../../02_Data/processed/final_all_results_fixed.csv", index=False, encoding="utf-8-sig")
print(f"\n🎉 재분류 완료! 'final_all_results_fixed.csv' 파일을 확인하세요.")

❌ 총 0개의 에러 데이터를 발견했습니다. 재시도 시작...


재분류 중: 0it [00:00, ?it/s]



🎉 재분류 완료! 'final_all_results_fixed.csv' 파일을 확인하세요.


In [ ]:
df_result = pd.read_csv("../../02_Data/processed/final_all_results.csv")

In [34]:
df_result

,thread_id,projectID,group,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes,category
0,1691589,5787,0,1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26,Praise_Support
1,1725920,5787,0,2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0,Product_Question
2,1725876,5787,0,1,about an hour left to go,0,1,1,1,0,1,Spam_Irrelevant
3,1725671,5787,0,1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3,Praise_Support
4,1725512,5787,0,1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5,Praise_Support
...,...,...,...,...,...,...,...,...,...,...,...,...
73434,133497,1143,1,3,"Hello. Will it be in Spanish? and if not, woul...",1,2,0,0,0,4,Product_Question
73435,133314,1143,1,2,Not sure if people get notifications for repli...,1,1,1,0,0,4,Product_Question
73436,133207,1143,1,2,The project looks interesting! Who is the desi...,1,0,0,0,0,0,Product_Question
73437,133159,1143,1,3,"Hi, is there the future possibility of a solo ...",1,1,0,0,0,11,Suggestion_Idea


In [11]:
target_df.head(10)

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes,category
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26,Praise_Support
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0,Product_Question
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,1,Spam_Irrelevant
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3,Praise_Support
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5,Praise_Support
5,5,5787,1724288,0,"[1724288, 1724568]",2,I was painting some seths and my mind wondered...,1,1,1,1,0,11,Suggestion_Idea
6,6,5787,1724185,0,[1724185],1,En que idiomas saldra?,0,0,0,0,0,0,Product_Question
7,7,5787,1723697,0,"[1723697, 1723864, 1724073]",3,Where do you guys produce the actual product i...,1,0,0,0,0,3,Product_Question
8,8,5787,1723483,0,"[1723483, 1724571]",2,One more to add as a game on my channel. Can't...,1,0,0,0,0,3,Praise_Support
9,9,5787,1723313,0,"[1723313, 1723863]",2,Excited for this one! Will there be the optio...,1,0,0,0,0,1,Product_Question


In [10]:
target_df['category'].value_counts()

category
Product_Question                                                     32538
Suggestion_Idea                                                      19122
Praise_Support                                                       10908
Shipping_Fulfillment                                                  5626
Complaint_Refund                                                      2984
Spam_Irrelevant                                                       2248
Please provide the thread text you would like me to analyze.             7
Please provide the [Thread Text] you would like me to analyze.           1
Please provide the thread text, and I will classify it for you.          1
Please provide the text you would like me to classify.                   1
Please provide the text you would like me to analyze.                    1
Please provide the full thread text you would like me to analyze.        1
Please provide the text you would like me to analyze!                    1
Name: count, dty

In [61]:
# 우리가 설정한 정상 카테고리 목록
valid_categories = [
    'Shipping_Fulfillment', 'Product_Question', 'Praise_Support', 
    'Complaint_Refund', 'Suggestion_Idea', 'Spam_Irrelevant'
]

# 'Error'이거나, 정상 카테고리에 포함되지 않는 모든 행을 추출
error_rows = df_result[~df_result['category'].isin(valid_categories)]

# 이상한 결과만 모아서 보여주기
print(f"🚨 총 {len(error_rows)}개의 이상 행을 발견했습니다.")
print(error_rows[['category', 'thread_text']].head(20))

# 따로 파일로 저장해서 엑셀로 편하게 확인하기
error_rows.to_csv("이상데이터_확인용.csv", index=False, encoding="utf-8-sig")
print("\n💾 '이상데이터_확인용.csv' 파일로 저장했습니다. 엑셀로 열어보세요!")

🚨 총 67개의 이상 행을 발견했습니다.
                                                category  \
1200   Please provide the text of the thread you woul...   
5943   Please provide the thread text you would like ...   
5950   Please provide the thread text you would like ...   
10345  Please provide the thread text you would like ...   
13200  Please provide the text you would like me to a...   
13674  Please provide the thread text you would like ...   
13709  Please provide the thread text you would like ...   
13755  Please provide the thread text you would like ...   
13886  Please provide the thread text so I can classi...   
24153  Please provide the text of the thread you woul...   
32389  Please provide the text you would like me to a...   
32778  Please provide the thread text you would like ...   
32852  Please provide the thread text you would like ...   
33089  Please provide the thread text you would like ...   
33407  Please provide the thread text you would like ...   
33451  Please pro

In [71]:
set(error_rows['projectID'])

{222,
 887,
 889,
 1032,
 2225,
 2460,
 2518,
 2757,
 3493,
 3848,
 4400,
 4475,
 4664,
 4874,
 5030,
 6626,
 6631,
 8203,
 8526}

In [83]:
len(df.loc[df['projectID']==8526])

864

In [ ]:
thread_df.loc[(df['projectID']==8526) & (df['comment_id']=='2595354')]

,projectID,comment_id,parent_id,depth_level,author_id,author_name,creator_id,is_pledge_master,is_backer,backer_number,...,is_pathfinder,has_children,children_count,text,created_at,likes,phaseLabel,campaignStart,campaignEnd,thread_id


In [103]:
thread_df.loc[(thread_df['projectID']==8526)& (thread_df['thread_id']=='2595354')]

,thread_id,projectID,thread_text,comment_count


In [96]:
df.columns

Index(['projectID', 'comment_id', 'parent_id', 'depth_level', 'author_id',
       'author_name', 'creator_id', 'is_pledge_master', 'is_backer',
       'backer_number', 'is_prior_backer', 'is_pathfinder', 'has_children',
       'children_count', 'text', 'created_at', 'likes', 'phaseLabel',
       'campaignStart', 'campaignEnd', 'thread_id'],
      dtype='object')

In [28]:
ttest_df=pd.read_csv('../../02_Data/raw/gamefound_488_comments.csv')

In [30]:
ttest_df

,project_ID,comment_id,parent_id,depth_level,author_id,author_name,creator_id,is_pledge_master,is_backer,backer_number,is_prior_backer,is_pathfinder,has_children,children_count,text,created_at,likes
0,5787,1691589,NaN,0,1456332,BlackSiteStudios,1,False,False,NaN,False,False,False,0,WELCOME TO EDEN!\n\nPlease check the FAQ and t...,2025-06-01T23:14:35.6Z,26
1,5787,2617125,NaN,0,1744120,TheRealNedry,0,False,True,1016.0,False,False,False,0,I didn't see the email until now and it needs ...,2026-06-04T02:04:40.78Z,0
2,5787,2616975,NaN,0,495054,tdlowe44,0,False,True,1038.0,False,False,True,1,I received the emails with the links. I clicke...,2026-06-03T22:44:14.413Z,0
3,5787,2617025,2616975.0,1,1456332,BlackSiteStudios,1,False,False,NaN,False,False,False,0,Hmm thats odd. The files we have on Drivethru ...,2026-06-03T23:48:01.403Z,0
4,5787,2607093,NaN,0,78722,SinisterHobby,0,False,True,NaN,False,False,False,0,Fingers crossed the physical book orders are a...,2026-05-29T18:16:59.173Z,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897051,1143,98174,NaN,0,457997,RiggedDiceGames,0,True,False,NaN,False,False,True,2,Hello! This game looks very interesting. Do yo...,2021-08-05T20:57:21.32Z,2
897052,1143,98616,98174.0,1,542570,Klabater,1,False,False,NaN,False,False,False,0,Hi there RiggedDiceGames. Currently TBC. Howev...,2021-08-06T09:21:26.337Z,2
897053,1143,99582,98174.0,1,457997,RiggedDiceGames,0,True,False,NaN,False,False,False,0,Looking forward to it!,2021-08-09T00:35:44.433Z,1
897054,1143,96836,NaN,0,49686,Mycha,0,False,False,NaN,False,False,True,1,"Please, do not let this game to be as unpolish...",2021-08-04T16:28:33.583Z,3


In [31]:
ttest_df.loc[ttest_df['project_ID']==1476]

,project_ID,comment_id,parent_id,depth_level,author_id,author_name,creator_id,is_pledge_master,is_backer,backer_number,is_prior_backer,is_pathfinder,has_children,children_count,text,created_at,likes
